# Февраль: 2 ИНН по каждой аномалии (Excel -> Озеро)

Тетрадка показывает для каждой аномалии по 2 ИНН:
1. сначала записи из Excel,
2. затем записи из Озера (ODS).

Аномалии:
- ИНН с несколькими договорами;
- ИНН с несколькими наименованиями;
- Наименование с несколькими ИНН;
- Смена договора в феврале (закрытие и открытие другого).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_inn_col = 'ИНН'
excel_contract_col = 'Номер договора'
excel_name_col = 'Наименование'

top_n_inn = 2

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_text(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = set([str(c).strip() for c in required_cols])
    for i in range(min(scan_rows, len(raw))):
        vals = set([str(x).strip() for x in raw.iloc[i].tolist()])
        if len(req & vals) >= 2:
            return i
    return 0

def to_sorted_top(vals, n=2):
    return sorted(list(vals))[:n]

## 0) Загрузка Excel и Озера

In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_contract_col, excel_name_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

excel_df = pd.DataFrame({
    'inn': excel_raw[excel_inn_col].apply(normalize_id),
    'contract_number': excel_raw[excel_contract_col].apply(normalize_text),
    'client_name': excel_raw[excel_name_col].apply(normalize_text),
})
excel_df = excel_df.dropna(subset=['inn']).drop_duplicates()

with imp:
    ods_all_raw = imp.fetch("""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where c.c_inn is not null
    """)

    ods_sa_active_raw = imp.fetch(f"""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and c.c_inn is not null
    """)

ods_all_df = pd.DataFrame({
    'inn': ods_all_raw['inn'].apply(normalize_id),
    'contract_number': ods_all_raw['contract_number'].apply(normalize_text),
    'client_name': ods_all_raw['client_name'].apply(normalize_text),
    'acq_class': ods_all_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_all_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_all_raw['d_valid_to'], errors='coerce').dt.date,
}).dropna(subset=['inn']).drop_duplicates()

ods_sa_active_df = pd.DataFrame({
    'inn': ods_sa_active_raw['inn'].apply(normalize_id),
    'contract_number': ods_sa_active_raw['contract_number'].apply(normalize_text),
    'client_name': ods_sa_active_raw['client_name'].apply(normalize_text),
    'acq_class': ods_sa_active_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_sa_active_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_sa_active_raw['d_valid_to'], errors='coerce').dt.date,
}).dropna(subset=['inn']).drop_duplicates()

display(pd.DataFrame([
    {'source': 'excel', 'rows': len(excel_df), 'unique_inn': excel_df['inn'].nunique(), 'header_row': header_row},
    {'source': 'ods_sa_active', 'rows': len(ods_sa_active_df), 'unique_inn': ods_sa_active_df['inn'].nunique(), 'header_row': None},
    {'source': 'ods_all', 'rows': len(ods_all_df), 'unique_inn': ods_all_df['inn'].nunique(), 'header_row': None},
]))

In [ ]:
def show_excel_then_lake(title, inn_candidates, lake_df, n=2):
    inn_pick = to_sorted_top(set(inn_candidates), n=n)
    print(f"\n=== {title} ===")
    print('selected_inn:', inn_pick)

    print('\nExcel rows:')
    display(
        excel_df[excel_df['inn'].isin(inn_pick)]
        .sort_values(['inn', 'contract_number', 'client_name'])
        .drop_duplicates()
    )

    print('\nLake rows:')
    display(
        lake_df[lake_df['inn'].isin(inn_pick)]
        .sort_values(['inn', 'contract_number', 'client_name'])
        .drop_duplicates()
    )

## 1) Аномалия: ИНН с несколькими договорами

In [ ]:
excel_multi_contract = (
    excel_df.assign(contract_key=excel_df['contract_number'].fillna('__NULL__'))
    .groupby('inn', as_index=False)
    .agg(contract_cnt=('contract_key', 'nunique'))
)
excel_multi_contract_inn = set(excel_multi_contract[excel_multi_contract['contract_cnt'] > 1]['inn'].tolist())

ods_multi_contract = (
    ods_sa_active_df.assign(contract_key=ods_sa_active_df['contract_number'].fillna('__NULL__'))
    .groupby('inn', as_index=False)
    .agg(contract_cnt=('contract_key', 'nunique'))
)
ods_multi_contract_inn = set(ods_multi_contract[ods_multi_contract['contract_cnt'] > 1]['inn'].tolist())

inn_union = sorted(list(excel_multi_contract_inn | ods_multi_contract_inn))
show_excel_then_lake('rule1_inn_multi_contracts', inn_union, ods_sa_active_df, n=top_n_inn)

## 2) Аномалия: ИНН с несколькими наименованиями

In [ ]:
excel_inn_multi_name = (
    excel_df.dropna(subset=['inn', 'client_name'])
    .groupby('inn', as_index=False)
    .agg(name_cnt=('client_name', 'nunique'))
)
excel_inn_multi_name_inn = set(excel_inn_multi_name[excel_inn_multi_name['name_cnt'] > 1]['inn'].tolist())

ods_inn_multi_name = (
    ods_sa_active_df.dropna(subset=['inn', 'client_name'])
    .groupby('inn', as_index=False)
    .agg(name_cnt=('client_name', 'nunique'))
)
ods_inn_multi_name_inn = set(ods_inn_multi_name[ods_inn_multi_name['name_cnt'] > 1]['inn'].tolist())

inn_union = sorted(list(excel_inn_multi_name_inn | ods_inn_multi_name_inn))
show_excel_then_lake('rule2_inn_multi_names', inn_union, ods_sa_active_df, n=top_n_inn)

## 3) Аномалия: наименование с несколькими ИНН

In [ ]:
excel_name_multi_inn = (
    excel_df.dropna(subset=['client_name', 'inn'])
    .groupby('client_name', as_index=False)
    .agg(inn_cnt=('inn', 'nunique'))
)
excel_bad_names = set(excel_name_multi_inn[excel_name_multi_inn['inn_cnt'] > 1]['client_name'].tolist())
excel_name_multi_inn_inn = set(excel_df[excel_df['client_name'].isin(excel_bad_names)]['inn'].dropna().tolist())

ods_name_multi_inn = (
    ods_sa_active_df.dropna(subset=['client_name', 'inn'])
    .groupby('client_name', as_index=False)
    .agg(inn_cnt=('inn', 'nunique'))
)
ods_bad_names = set(ods_name_multi_inn[ods_name_multi_inn['inn_cnt'] > 1]['client_name'].tolist())
ods_name_multi_inn_inn = set(ods_sa_active_df[ods_sa_active_df['client_name'].isin(ods_bad_names)]['inn'].dropna().tolist())

inn_union = sorted(list(excel_name_multi_inn_inn | ods_name_multi_inn_inn))
show_excel_then_lake('rule2_names_multi_inn', inn_union, ods_sa_active_df, n=top_n_inn)

## 4) Аномалия: смена договора в феврале (закрыт один, открыт другой)

In [ ]:
month_start_dt = pd.to_datetime(month_start).date()
month_end_dt = pd.to_datetime(month_end).date()

closed_in_feb = ods_all_df[
    ods_all_df['d_valid_to'].notna() &
    (ods_all_df['d_valid_to'] >= month_start_dt) &
    (ods_all_df['d_valid_to'] <= month_end_dt)
][['inn', 'contract_number', 'd_valid_to']].drop_duplicates()

opened_in_feb = ods_all_df[
    ods_all_df['d_valid_from'].notna() &
    (ods_all_df['d_valid_from'] >= month_start_dt) &
    (ods_all_df['d_valid_from'] <= month_end_dt)
][['inn', 'contract_number', 'd_valid_from']].drop_duplicates()

switch_df = closed_in_feb.merge(
    opened_in_feb[['inn', 'contract_number', 'd_valid_from']],
    on='inn', how='inner', suffixes=('_closed', '_opened')
)
switch_df = switch_df[switch_df['contract_number_closed'] != switch_df['contract_number_opened']]
switch_inn = set(switch_df['inn'].dropna().tolist())

show_excel_then_lake('rule4_contract_switch_within_feb', switch_inn, ods_all_df, n=top_n_inn)

print('\nSwitch details for selected INN:')
display(switch_df[switch_df['inn'].isin(to_sorted_top(switch_inn, top_n_inn))].head(100))

## 5) Проверка совпадения ИНН клиентов эквайринга (Excel vs Озеро, февраль)

Фильтры Озера сохраняются: используется `ods_sa_active_df` (только `SA` active на `2026-02-01`).

Ниже:
- сводка пересечения множеств ИНН;
- строки по 2 ИНН, которые есть только в Excel;
- строки по 2 ИНН, которые есть только в Озере.

In [ ]:
excel_inn_set = set(excel_df['inn'].dropna().tolist())
lake_inn_set = set(ods_sa_active_df['inn'].dropna().tolist())

common_inn = excel_inn_set & lake_inn_set
only_excel_inn = sorted(list(excel_inn_set - lake_inn_set))
only_lake_inn = sorted(list(lake_inn_set - excel_inn_set))

inn_match_summary = pd.DataFrame([{
    'excel_unique_inn': len(excel_inn_set),
    'lake_unique_inn': len(lake_inn_set),
    'common_inn': len(common_inn),
    'only_excel_inn': len(only_excel_inn),
    'only_lake_inn': len(only_lake_inn),
}])

display(inn_match_summary)

pick_only_excel = only_excel_inn[:2]
pick_only_lake = only_lake_inn[:2]

print('selected only_excel_inn:', pick_only_excel)
print('selected only_lake_inn:', pick_only_lake)

print('\nRows for only_excel INN (Excel first):')
display(
    excel_df[excel_df['inn'].isin(pick_only_excel)]
    .sort_values(['inn', 'contract_number', 'client_name'])
    .drop_duplicates()
)
print('Rows for only_excel INN (Lake):')
display(
    ods_sa_active_df[ods_sa_active_df['inn'].isin(pick_only_excel)]
    .sort_values(['inn', 'contract_number', 'client_name'])
    .drop_duplicates()
)

print('\nRows for only_lake INN (Excel):')
display(
    excel_df[excel_df['inn'].isin(pick_only_lake)]
    .sort_values(['inn', 'contract_number', 'client_name'])
    .drop_duplicates()
)
print('Rows for only_lake INN (Lake first):')
display(
    ods_sa_active_df[ods_sa_active_df['inn'].isin(pick_only_lake)]
    .sort_values(['inn', 'contract_number', 'client_name'])
    .drop_duplicates()
)